In [1]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import deque
import random
from tqdm import tqdm
import matplotlib.pyplot as plt

In [2]:
class ReplayMemory:
    def __init__(self, maxlen):
        self.buffer = deque(maxlen=maxlen)

    def add(self, state, action, reward, next_state, terminated):
        self.buffer.append((state, action, reward, next_state, terminated))

    def size(self):
        return len(self.buffer)

    def sample(self, size):
        transitions = random.sample(self.buffer, k=size)
        states, actions, rewards, next_states, terminated = zip(*transitions)
        return np.array(states), np.array(actions), rewards, np.array(next_states), terminated

In [3]:
class PolicyNet(nn.Module):
    def __init__(self, state_dim, hidden_dim, action_dim, action_bound):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, action_dim)
        self.action_bound = action_bound

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.tanh(self.fc2(x)) * self.action_bound
        return x


class QValueNet(nn.Module):
    def __init__(self, state_dim, hidden_dim, action_dim):  # Takes in state and action, outputs q value
        super().__init__()
        self.fc1 = nn.Linear(state_dim + action_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)

    def forward(self, s, a):
        x = torch.concat([s, a], dim=-1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


In [4]:
class DDPG:
    def __init__(self, state_dim, hidden_dim, action_dim, action_bound, actor_lr, critic_lr, epochs, sigma, gamma, tau, device):
        self.actor = PolicyNet(state_dim, hidden_dim, action_dim, action_bound).to(device)
        self.critic = QValueNet(state_dim, hidden_dim, action_dim).to(device)
        self.actor_target = PolicyNet(state_dim, hidden_dim, action_dim, action_bound).to(device)
        self.critic_target = QValueNet(state_dim, hidden_dim, action_dim).to(device)
        self.actor_target.load_state_dict(self.actor.state_dict())
        self.critic_target.load_state_dict(self.critic.state_dict())
        self.actor_optimizer = torch.optim.Adam(self.actor.parameters(), lr=actor_lr)
        self.critic_optimizer = torch.optim.Adam(self.critic.parameters(), lr=critic_lr)

        self.epochs = epochs
        self.sigma = sigma  # std for the gaussian noise
        self.gamma = gamma
        self.tau = tau  # polyak averaging constant
        self.device = device

    def take_action(self, state):
        state = torch.tensor(state, dtype=torch.float).to(self.device)
        action = self.actor(state)
        noise = torch.normal(torch.zeros_like(action), torch.ones_like(action) * self.sigma)
        return np.array((action + noise).detach().cpu())

    def soft_update(self, target_net, net):  # polyak averaging
        for target_param, param in zip(target_net.parameters(), net.parameters()):
            target_param.data.copy_(target_param * (1 - self.tau) + param * self.tau)

    def update(self, transition_dict):
        states = torch.tensor(transition_dict['states'], dtype=torch.float).to(self.device)
        actions = torch.tensor(transition_dict['actions'], dtype=torch.float).view(-1, 1).to(self.device)
        rewards = torch.tensor(transition_dict['rewards'], dtype=torch.float).view(-1, 1).to(self.device)
        next_states = torch.tensor(transition_dict['next_states'], dtype=torch.float).to(self.device)
        terminated = torch.tensor(transition_dict['terminated'], dtype=torch.float).view(-1, 1).to(self.device)

        for _ in range(self.epochs):
            targets = rewards + self.gamma * self.critic_target(next_states, self.actor_target(next_states)).detach() * (1 - terminated)  # The reason we need targets for both critic and actor is because we want this target to stay stable
            critic_loss = F.mse_loss(targets, self.critic(states, actions))
            actor_loss = -torch.mean(self.critic(states, self.actor(states)))

            self.critic_optimizer.zero_grad()
            critic_loss.backward()
            self.critic_optimizer.step()
            self.actor_optimizer.zero_grad()
            actor_loss.backward()
            self.actor_optimizer.step()

            self.soft_update(self.critic_target, self.critic)
            self.soft_update(self.actor_target, self.actor)


In [5]:
num_episodes = 300
hidden_dim = 128
actor_lr = 3e-4
critic_lr = 3e-3
epochs = 10
sigma = 0.01
gamma = 0.98
tau = 0.005
buffer_size = 50000
minimum_size = 5000
batch_size = 64
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

buffer = ReplayMemory(buffer_size)

env_name = 'Pendulum-v1'
env = gym.make(env_name)
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]
action_bound = env.action_space.high[0]

agent = DDPG(state_dim, hidden_dim,action_dim,action_bound, actor_lr, critic_lr, epochs, sigma, gamma, tau, device)

env.reset(seed=0)
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)


In [6]:
return_list = []
for i in range(10):
    with tqdm(total=num_episodes // 10, desc="Iteration %d" % i) as pbar:
        for i_episode in range(num_episodes // 10):
            episode_return = 0
            state, info = env.reset()
            while True:
                action = agent.take_action(state)
                next_state, reward, terminated, truncated, info = env.step(action)
                buffer.add(state, action, reward, next_state, terminated)
                if buffer.size() > minimum_size:
                    s, a, r, ns, t = buffer.sample(batch_size)
                    transition_dict = {'states': s, 'actions': a, 'rewards': r, 'next_states': ns, 'terminated': t}
                    agent.update(transition_dict)
                episode_return += reward
                state = next_state
                if terminated or truncated:
                    break
            if (i_episode + 1) % 10 == 0:
                pbar.set_postfix({'episode': '%d' % (num_episodes / 10 * i + i_episode + 1),
                                      'return': '%.3f' % np.mean(return_list[-10:])})
            pbar.update(1)

Iteration 0:   0%|          | 0/30 [00:00<?, ?it/s]/var/folders/c9/lk6xzy711r9dzmt0855740gm0000gn/T/ipykernel_3410/1684414628.py:22: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  return np.array((action + noise).detach().cpu())
Iteration 0:  30%|███       | 9/30 [00:01<00:02,  7.03it/s]/opt/anaconda3/envs/RL/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/anaconda3/envs/RL/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
Iteration 0:  83%|████████▎ | 25/30 [00:04<00:00,  5.94it/s, episode=20, return=nan]


RuntimeError: one of the variables needed for gradient computation has been modified by an inplace operation: [MPSFloatType [1, 128]] is at version 2; expected version 1 instead. Hint: enable anomaly detection to find the operation that failed to compute its gradient, with torch.autograd.set_detect_anomaly(True).

In [ ]:
episode_list = list(range(len(return_list)))
plt.plot(episode_list, return_list)
plt.xlabel('Episodes')
plt.ylabel('Returns')
plt.title('DDPG on Pendulum-v1')
plt.show()

def moving_average(a, window_size):
    cumulative_sum = np.cumsum(np.insert(a, 0, 0))
    middle = (cumulative_sum[window_size:] - cumulative_sum[:-window_size]) / window_size

    r = np.arange(1, window_size, 2)
    begin = np.cumsum(a[:window_size-1])[::2] / r
    end = (np.cumsum(a[:-window_size:-1])[::2] / r)[::-1]
    return np.concatenate((begin, middle, end))

mv_return = moving_average(return_list, 9)
plt.plot(episode_list, mv_return)
plt.xlabel('Episodes')
plt.ylabel('Returns')
plt.title('DDPG on Pendulum-v1')
plt.show()
